In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix
from xgboost import XGBClassifier
import joblib

sns.set_theme(style='whitegrid')
PROCESSED = Path('../data/processed')
MODELS    = Path('../models')

In [ ]:
df = pd.read_csv(PROCESSED / 'model_features.csv')
print(df.shape)
print(df['result'].value_counts())

In [ ]:
# elo_diff is NaN for non-WC teams so we fill with 0 (neutral — no info)
df['elo_diff'] = df['elo_diff'].fillna(0)

features = [
    'rank_diff',
    'point_diff',
    'elo_diff',
    'is_friendly',
    'goals_scored_home',
    'goals_conceded_home',
    'goals_scored_away',
    'goals_conceded_away',
]

X = df[features]
y = df['result']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'train: {len(X_train)}  test: {len(X_test)}')

In [ ]:
rf = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42)
rf.fit(X_train, y_train)

rf_score = rf.score(X_test, y_test)
print(f'random forest test accuracy: {rf_score:.1%}')

rf_cv = cross_val_score(rf, X, y, cv=5).mean()
print(f'random forest 5-fold cv accuracy: {rf_cv:.1%}')

In [ ]:
label_map   = {-1: 0, 0: 1, 1: 2}
label_unmap = {0: -1, 1: 0, 2: 1}

y_train_xgb = y_train.map(label_map)
y_test_xgb  = y_test.map(label_map)

xgb = XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.05,
                    eval_metric='mlogloss', random_state=42)
xgb.fit(X_train, y_train_xgb)

xgb_score = xgb.score(X_test, y_test_xgb)
print(f'xgboost test accuracy: {xgb_score:.1%}')

y_xgb = y.map(label_map)
xgb_cv = cross_val_score(xgb, X, y_xgb, cv=5).mean()
print(f'xgboost 5-fold cv accuracy: {xgb_cv:.1%}')

In [ ]:
rf_preds = rf.predict(X_test)
cm = confusion_matrix(y_test, rf_preds, labels=[1, 0, -1])

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Home Win', 'Draw', 'Away Win'],
            yticklabels=['Home Win', 'Draw', 'Away Win'])
plt.ylabel('actual')
plt.xlabel('predicted')
plt.title('random forest confusion matrix')
plt.tight_layout()
plt.show()

In [ ]:
print(classification_report(y_test, rf_preds, target_names=['Away Win', 'Draw', 'Home Win']))

In [ ]:
importances = pd.Series(rf.feature_importances_, index=features).sort_values(ascending=True)

importances.plot(kind='barh', figsize=(8, 5), color='steelblue')
plt.xlabel('importance')
plt.title('random forest feature importance')
plt.tight_layout()
plt.show()

In [ ]:
joblib.dump(rf,  MODELS / 'random_forest.pkl')
joblib.dump(xgb, MODELS / 'xgboost.pkl')
print('models saved')